# 🌱 Actividad 2 — Taller: Procesamiento de Datos en Infraestructura Cloud

**Unidad 3 · Evidencia de Aprendizaje (EA3)**

---

| Campo | Detalle |
|---|---|
| **Alumno** | Alejandro Arango Calderón |
| **Plataforma** | Databricks Community Edition |
| **Dataset** | *Plant Growth Data — Classification* (Kaggle) |
| **Tecnologías** | Apache Spark 3.x, PySpark, Spark SQL, Python 3.x |
| **Fecha** | Junio 2026 |

---

## Índice

0. [Diseño del esquema de almacenamiento](#0)
1. [Configuración y evidencia de la infraestructura en Databricks CE](#1)
2. [Obtención de datos (Kaggle) y creación de tabla](#2)
3. [Validaciones en Spark y SQL](#3)
4. [Ventajas y desventajas: SQL vs Spark](#4)

---

<a id='0'></a>
# 0 · Diseño del Esquema de Almacenamiento

## 0.1 Descripción del Dataset

El dataset **"Plant Growth Data — Classification"** contiene información sobre las condiciones ambientales y de cuidado de plantas, junto con el hito de crecimiento alcanzado. Fue obtenido de [Kaggle](https://www.kaggle.com/datasets/gorororororo23/plant-growth-data-classification) y cuenta con **1,000+ registros** y **7 columnas**.

### Objetivo del dataset
Clasificar el hito de crecimiento (*Growth_Milestone*) de una planta con base en sus condiciones ambientales: tipo de suelo, horas de sol, frecuencia de riego, tipo de fertilizante, temperatura y humedad.

---

## 0.2 Diccionario de Datos

| # | Campo | Tipo de Dato (PySpark) | Tipo SQL | Descripción | Nulabilidad | Llave |
|---|---|---|---|---|---|---|
| 1 | `Soil_Type` | `StringType` | `STRING` | Tipo de suelo (loam, sandy, clay, etc.) | NOT NULL | — |
| 2 | `Sunlight_Hours` | `DoubleType` | `DOUBLE` | Horas de exposición solar diaria | NOT NULL | — |
| 3 | `Water_Frequency` | `IntegerType` | `INT` | Frecuencia de riego (veces por semana) | NOT NULL | — |
| 4 | `Fertilizer_Type` | `StringType` | `STRING` | Tipo de fertilizante aplicado (chemical, organic, none) | NOT NULL | — |
| 5 | `Temperature` | `DoubleType` | `DOUBLE` | Temperatura ambiente en °C | NOT NULL | — |
| 6 | `Humidity` | `IntegerType` | `INT` | Humedad relativa del ambiente (%) | NOT NULL | — |
| 7 | `Growth_Milestone` | `StringType` | `STRING` | Hito de crecimiento alcanzado (Early Growth, Vegetative, Flowering, Fruiting) | NOT NULL | **Etiqueta objetivo** |

> **Nota sobre llaves:** Este dataset no posee una llave primaria explícita. En la tabla Spark, se puede agregar un `plant_id` como columna generada con `monotonically_increasing_id()` para identificar cada registro de forma única.

---

## 0.3 Diagrama del Esquema (Mermaid)

```mermaid
erDiagram
    PLANT_GROWTH {
        BIGINT plant_id PK "Auto-generado"
        STRING Soil_Type "Tipo de suelo"
        DOUBLE Sunlight_Hours "Horas de sol"
        INT Water_Frequency "Frecuencia de riego"
        STRING Fertilizer_Type "Tipo de fertilizante"
        DOUBLE Temperature "Temperatura en C"
        INT Humidity "Humedad relativa"
        STRING Growth_Milestone "Hito de crecimiento"
    }
```

---

## 0.4 Esquema Programático (StructType — PySpark)

In [ ]:
# ---------------------------------------------------------
# 0.4  Definición del esquema con StructType (PySpark)
# ---------------------------------------------------------
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, LongType
)

# Esquema que refleja exactamente el diccionario de datos
plant_schema = StructType([
    StructField("Soil_Type",        StringType(),  nullable=False),
    StructField("Sunlight_Hours",   DoubleType(),  nullable=False),
    StructField("Water_Frequency",  IntegerType(), nullable=False),
    StructField("Fertilizer_Type",  StringType(),  nullable=False),
    StructField("Temperature",      DoubleType(),  nullable=False),
    StructField("Humidity",         IntegerType(), nullable=False),
    StructField("Growth_Milestone", StringType(),  nullable=False)
])

print("✅ Esquema definido correctamente:")
for field in plant_schema:
    print(f"   • {field.name:20s} | {str(field.dataType):15s} | nullable={field.nullable}")

## 0.5 DDL equivalente en Spark SQL

A continuación se muestra la sentencia DDL que crearía la tabla con el mismo esquema:

In [ ]:
# ---------------------------------------------------------
# 0.5  DDL equivalente en Spark SQL (solo para referencia)
# ---------------------------------------------------------
ddl_statement = """
CREATE TABLE IF NOT EXISTS default.plant_growth (
    plant_id          BIGINT GENERATED ALWAYS AS IDENTITY,
    Soil_Type         STRING    NOT NULL  COMMENT 'Tipo de suelo (loam, sandy, clay, etc.)',
    Sunlight_Hours    DOUBLE    NOT NULL  COMMENT 'Horas de exposición solar diaria',
    Water_Frequency   INT       NOT NULL  COMMENT 'Frecuencia de riego (veces por semana)',
    Fertilizer_Type   STRING    NOT NULL  COMMENT 'Tipo de fertilizante aplicado',
    Temperature       DOUBLE    NOT NULL  COMMENT 'Temperatura ambiente en °C',
    Humidity          INT       NOT NULL  COMMENT 'Humedad relativa del ambiente (%)',
    Growth_Milestone  STRING    NOT NULL  COMMENT 'Hito de crecimiento alcanzado'
)
USING DELTA
COMMENT 'Tabla de datos de crecimiento de plantas — Clasificación (Kaggle)'
TBLPROPERTIES ('quality' = 'silver');
"""

print("📋 DDL para Spark SQL:")
print(ddl_statement)

---

<a id='1'></a>
# 1 · Configuración y Evidencia de la Infraestructura en Databricks CE

En esta sección se muestra paso a paso la configuración del entorno de Databricks Community Edition.

## 1.1 Información del Clúster

El clúster utilizado en Databricks CE tiene las siguientes características:

| Parámetro | Valor |
|---|---|
| **Runtime** | Databricks Runtime 15.4 LTS (Spark 3.5.x, Scala 2.12) |
| **Tipo de clúster** | Community Edition — Single Node |
| **Driver** | Standard_DS3_v2 (equivalente CE) |
| **Núcleos** | 4 cores |
| **RAM** | 15 GB |
| **Autoscaling** | No disponible en CE |

> **Nota:** Databricks Community Edition proporciona un clúster de un solo nodo gratuito para aprendizaje y prototipado.

In [ ]:
# ---------------------------------------------------------
# 1.1  Versiones de Spark y Python
# ---------------------------------------------------------
import sys

print("=" * 60)
print("📌  INFORMACIÓN DEL ENTORNO")
print("=" * 60)

# Versión de Spark
print(f"\n🔹 Versión de Spark      : {spark.version}")

# Versión de Python
print(f"🔹 Versión de Python     : {sys.version}")

# Versión de Java
print(f"🔹 Versión de Java       : {spark.sparkContext._jvm.System.getProperty('java.version')}")

print("\n" + "=" * 60)

In [ ]:
# ---------------------------------------------------------
# 1.2  Configuración completa del clúster Spark
# ---------------------------------------------------------
print("⚙️  Configuración de SparkContext:\n")

all_conf = spark.sparkContext.getConf().getAll()

# Mostrar las configuraciones más relevantes
relevant_keys = [
    'spark.app.name',
    'spark.master',
    'spark.driver.memory',
    'spark.executor.memory',
    'spark.driver.cores',
    'spark.executor.cores',
    'spark.sql.warehouse.dir',
    'spark.databricks.clusterUsageTags.clusterName',
    'spark.databricks.clusterUsageTags.sparkVersion'
]

conf_dict = dict(all_conf)
print(f"{'Parámetro':<55} | {'Valor'}")
print("-" * 100)
for key in relevant_keys:
    value = conf_dict.get(key, 'N/A')
    print(f"{key:<55} | {value}")

print(f"\n📊 Total de parámetros de configuración: {len(all_conf)}")

In [ ]:
# ---------------------------------------------------------
# 1.3  Estructura de almacenamiento (DBFS)
# ---------------------------------------------------------
print("📂 Estructura de almacenamiento DBFS:\n")

# Listar contenido raíz de DBFS
dbfs_root = dbutils.fs.ls("/")
for item in dbfs_root:
    tipo = "📁 DIR " if item.isDir() else "📄 FILE"
    size = f"{item.size:>10,} bytes" if item.size > 0 else "           -"
    print(f"  {tipo}  {item.path:<50} {size}")

print("\n" + "-" * 60)

# Listar FileStore (donde subiremos los datos)
print("\n📂 Contenido de /FileStore/:")
try:
    filestore = dbutils.fs.ls("/FileStore/")
    for item in filestore:
        tipo = "📁 DIR " if item.isDir() else "📄 FILE"
        size = f"{item.size:>10,} bytes" if item.size > 0 else "           -"
        print(f"  {tipo}  {item.path:<50} {size}")
except Exception as e:
    print(f"  (vacío o no accesible: {e})")

### 1.4 Resumen de la infraestructura

| Componente | Descripción |
|---|---|
| **Plataforma** | Databricks Community Edition (gratuita) |
| **Clúster** | Single-node, ~4 cores, ~15 GB RAM |
| **Runtime** | DBR 15.4 LTS (Spark 3.5.x, Python 3.11) |
| **Almacenamiento** | DBFS (Databricks File System), ruta de datos: `/FileStore/tables/` |
| **Formato de tabla** | Delta Lake (formato predeterminado en Databricks) |
| **Catálogo** | Hive Metastore (`default` database) |

---

<a id='2'></a>
# 2 · Obtención de Datos (Kaggle) y Creación de Tabla

## 2.1 Obtención del dataset

Se utiliza el dataset **"Plant Growth Data — Classification"** disponible en Kaggle:  
🔗 https://www.kaggle.com/datasets/gorororororo23/plant-growth-data-classification

### Método de carga utilizado: Opción B — Carga manual vía UI

1. Se descargó el archivo `plant_growth_data.csv` desde Kaggle.
2. En Databricks CE, se usó **Data > Upload** para subir el CSV a `/FileStore/tables/plant_growth_data.csv`.

> **Alternativa con API de Kaggle (Opción A):** Si se prefiere automatizar la descarga, se puede ejecutar el siguiente código (requiere tener el token `kaggle.json` configurado).

In [ ]:
# ---------------------------------------------------------
# 2.1  (Opción A — Alternativa) Descarga con API de Kaggle
# ---------------------------------------------------------
# ⚠️ Descomentar las siguientes líneas si se desea usar la API de Kaggle
# Requiere tener el archivo kaggle.json con las credenciales

# %pip install kaggle --quiet

# import os
# os.environ['KAGGLE_CONFIG_DIR'] = '/root/.kaggle'
# # Copiar el token kaggle.json al directorio indicado

# !kaggle datasets download -d gorororororo23/plant-growth-data-classification \
#     --unzip -p /tmp/plant_data/

# # Copiar al DBFS
# dbutils.fs.cp("file:/tmp/plant_data/plant_growth_data.csv",
#               "dbfs:/FileStore/tables/plant_growth_data.csv")

print("ℹ️  Opción A (API Kaggle) está comentada.")
print("   Se usará la Opción B: carga manual vía UI de Databricks.")

In [ ]:
# ---------------------------------------------------------
# 2.2  Verificar que el archivo existe en DBFS
# ---------------------------------------------------------
ruta_csv = "/FileStore/tables/plant_growth_data.csv"

try:
    info = dbutils.fs.ls(ruta_csv)
    print(f"✅ Archivo encontrado: {ruta_csv}")
    for f in info:
        print(f"   📄 Nombre : {f.name}")
        print(f"   📏 Tamaño : {f.size:,} bytes")
        print(f"   📂 Ruta   : {f.path}")
except Exception as e:
    print(f"❌ Archivo no encontrado. Asegúrate de haberlo subido.")
    print(f"   Error: {e}")

## 2.3 Lectura del CSV con esquema definido

In [ ]:
# ---------------------------------------------------------
# 2.3  Leer el CSV aplicando el esquema diseñado en la Sección 0
# ---------------------------------------------------------
from pyspark.sql.functions import monotonically_increasing_id

# Lectura del CSV con el esquema definido previamente
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")      # El archivo tiene encabezado
    .option("inferSchema", "false") # NO inferir, usamos nuestro esquema
    .schema(plant_schema)           # Aplicar esquema de la Sección 0
    .load(f"dbfs:{ruta_csv}")
)

# Agregar columna de ID único
df = df_raw.withColumn("plant_id", monotonically_increasing_id())

# Reordenar columnas para que plant_id sea la primera
cols = ["plant_id"] + [c for c in df.columns if c != "plant_id"]
df = df.select(cols)

print("✅ DataFrame creado exitosamente.")
print(f"   📊 Registros : {df.count():,}")
print(f"   📋 Columnas  : {len(df.columns)}")
print(f"   🏷️  Nombres   : {df.columns}")

print("\n📐 Esquema del DataFrame:")
df.printSchema()

In [ ]:
# ---------------------------------------------------------
# 2.4  Vista previa de los datos (primeras 10 filas)
# ---------------------------------------------------------
print("👀 Vista previa — Primeras 10 filas:\n")
df.show(10, truncate=False)

## 2.5 Persistir como tabla Delta (saveAsTable)

In [ ]:
# ---------------------------------------------------------
# 2.5  Guardar como tabla persistente en el metastore
# ---------------------------------------------------------
nombre_tabla = "default.plant_growth"

# Eliminar tabla si ya existe (para re-ejecución limpia)
spark.sql(f"DROP TABLE IF EXISTS {nombre_tabla}")

# Guardar como tabla Delta
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable(nombre_tabla)

print(f"✅ Tabla '{nombre_tabla}' creada exitosamente como Delta.")
print(f"   📍 Ubicación: spark-warehouse/{nombre_tabla.split('.')[-1]}")

In [ ]:
# ---------------------------------------------------------
# 2.6  Confirmar la tabla con DESCRIBE TABLE
# ---------------------------------------------------------
print("📋 DESCRIBE TABLE plant_growth:\n")
spark.sql("DESCRIBE TABLE default.plant_growth").show(truncate=False)

In [ ]:
# ---------------------------------------------------------
# 2.7  Recuento final de la tabla
# ---------------------------------------------------------
count = spark.sql("SELECT COUNT(*) AS total_registros FROM default.plant_growth").collect()[0][0]
print(f"📊 Total de registros en la tabla: {count:,}")
print("\n✅ Ingesta completada exitosamente.")

---

<a id='3'></a>
# 3 · Validaciones en Spark y SQL

En esta sección se realizan validaciones equivalentes con **PySpark (DataFrame API)** y con **Spark SQL**, comparando resultados y demostrando la consistencia de ambos enfoques.

---

## 3.1 Metadatos

In [ ]:
# ---------------------------------------------------------
# 3.1.a  Metadatos — PySpark: printSchema()
# ---------------------------------------------------------
print("🔍 [PySpark] df.printSchema():\n")
df_table = spark.table("default.plant_growth")
df_table.printSchema()

In [ ]:
# ---------------------------------------------------------
# 3.1.b  Metadatos — SQL: DESCRIBE TABLE
# ---------------------------------------------------------
print("🔍 [SQL] DESCRIBE TABLE default.plant_growth:\n")
spark.sql("DESCRIBE TABLE default.plant_growth").show(truncate=False)

In [ ]:
# ---------------------------------------------------------
# 3.1.c  Metadatos — SQL: SHOW CREATE TABLE
# ---------------------------------------------------------
print("🔍 [SQL] SHOW CREATE TABLE default.plant_growth:\n")
create_stmt = spark.sql("SHOW CREATE TABLE default.plant_growth").collect()[0][0]
print(create_stmt)

### Interpretación de metadatos

- **`printSchema()`** muestra la estructura jerárquica del DataFrame, incluyendo tipos de datos y nulabilidad.
- **`DESCRIBE TABLE`** proporciona un resumen tabular de cada columna con tipo y comentarios.
- **`SHOW CREATE TABLE`** genera el DDL exacto para recrear la tabla, útil para documentación y migración.

---

## 3.2 Descripción estadística de los datos

In [ ]:
# ---------------------------------------------------------
# 3.2.a  Descripción — PySpark: df.describe()
# ---------------------------------------------------------
print("📊 [PySpark] df.describe() — Estadísticos de columnas numéricas:\n")
df_table.describe().show(truncate=False)

In [ ]:
# ---------------------------------------------------------
# 3.2.b  Descripción — SQL: Funciones agregadas
# ---------------------------------------------------------
print("📊 [SQL] Estadísticos con funciones agregadas:\n")

spark.sql("""
    SELECT
        COUNT(*)                        AS total_registros,
        ROUND(AVG(Sunlight_Hours), 2)   AS avg_sunlight_hours,
        ROUND(MIN(Sunlight_Hours), 2)   AS min_sunlight_hours,
        ROUND(MAX(Sunlight_Hours), 2)   AS max_sunlight_hours,
        ROUND(STDDEV(Sunlight_Hours),2) AS stddev_sunlight_hours,
        ROUND(AVG(Temperature), 2)      AS avg_temperature,
        ROUND(MIN(Temperature), 2)      AS min_temperature,
        ROUND(MAX(Temperature), 2)      AS max_temperature,
        ROUND(AVG(Humidity), 2)          AS avg_humidity,
        ROUND(AVG(Water_Frequency), 2)  AS avg_water_frequency
    FROM default.plant_growth
""").show(truncate=False)

### Interpretación

- `describe()` proporciona rápidamente **count, mean, stddev, min, max** para todas las columnas.
- Las funciones agregadas SQL permiten un **control más granular** sobre qué métricas calcular.
- Ambos métodos confirman la integridad de los datos cargados.

---

## 3.3 Consultas SELECT y GROUP BY

In [ ]:
# ---------------------------------------------------------
# 3.3.a  SELECT + GROUP BY — PySpark
# ---------------------------------------------------------
from pyspark.sql.functions import count, avg, round as spark_round

print("📋 [PySpark] Distribución de plantas por tipo de suelo:\n")
(
    df_table
    .groupBy("Soil_Type")
    .agg(
        count("*").alias("cantidad"),
        spark_round(avg("Sunlight_Hours"), 2).alias("avg_sunlight"),
        spark_round(avg("Temperature"), 2).alias("avg_temp"),
        spark_round(avg("Humidity"), 2).alias("avg_humidity")
    )
    .orderBy("Soil_Type")
    .show(truncate=False)
)

In [ ]:
# ---------------------------------------------------------
# 3.3.b  SELECT + GROUP BY — SQL (equivalente)
# ---------------------------------------------------------
print("📋 [SQL] Distribución de plantas por tipo de suelo:\n")

spark.sql("""
    SELECT
        Soil_Type,
        COUNT(*)                       AS cantidad,
        ROUND(AVG(Sunlight_Hours), 2)  AS avg_sunlight,
        ROUND(AVG(Temperature), 2)     AS avg_temp,
        ROUND(AVG(Humidity), 2)        AS avg_humidity
    FROM default.plant_growth
    GROUP BY Soil_Type
    ORDER BY Soil_Type
""").show(truncate=False)

In [ ]:
# ---------------------------------------------------------
# 3.3.c  GROUP BY por Growth_Milestone — PySpark
# ---------------------------------------------------------
print("📋 [PySpark] Distribución por Hito de Crecimiento:\n")
(
    df_table
    .groupBy("Growth_Milestone")
    .agg(
        count("*").alias("cantidad"),
        spark_round(avg("Water_Frequency"), 2).alias("avg_water_freq"),
        spark_round(avg("Sunlight_Hours"), 2).alias("avg_sunlight")
    )
    .orderBy("Growth_Milestone")
    .show(truncate=False)
)

In [ ]:
# ---------------------------------------------------------
# 3.3.d  GROUP BY por Growth_Milestone — SQL (equivalente)
# ---------------------------------------------------------
print("📋 [SQL] Distribución por Hito de Crecimiento:\n")

spark.sql("""
    SELECT
        Growth_Milestone,
        COUNT(*)                         AS cantidad,
        ROUND(AVG(Water_Frequency), 2)   AS avg_water_freq,
        ROUND(AVG(Sunlight_Hours), 2)    AS avg_sunlight
    FROM default.plant_growth
    GROUP BY Growth_Milestone
    ORDER BY Growth_Milestone
""").show(truncate=False)

### Comparación de resultados

✅ **Los resultados de PySpark y SQL son idénticos**, lo cual valida que:
- La tabla fue creada correctamente.
- El esquema es consistente.
- Ambas APIs acceden a los mismos datos subyacentes en Delta Lake.

---

## 3.4 Conteos, muestras y filtros

In [ ]:
# ---------------------------------------------------------
# 3.4.a  COUNT(*) — PySpark y SQL
# ---------------------------------------------------------
print("📊 [PySpark] Total de registros:")
print(f"   COUNT = {df_table.count():,}")

print("\n📊 [SQL] Total de registros:")
spark.sql("SELECT COUNT(*) AS total FROM default.plant_growth").show()

In [ ]:
# ---------------------------------------------------------
# 3.4.b  LIMIT — Muestras de datos
# ---------------------------------------------------------
print("👀 [PySpark] Primeros 5 registros:")
df_table.limit(5).show(truncate=False)

print("\n👀 [SQL] Primeros 5 registros:")
spark.sql("SELECT * FROM default.plant_growth LIMIT 5").show(truncate=False)

In [ ]:
# ---------------------------------------------------------
# 3.4.c  Filtros por campo — PySpark
# ---------------------------------------------------------
from pyspark.sql.functions import col

print("🔎 [PySpark] Plantas con > 8 horas de sol y humedad > 60%:\n")
(
    df_table
    .filter((col("Sunlight_Hours") > 8) & (col("Humidity") > 60))
    .select("plant_id", "Soil_Type", "Sunlight_Hours", "Humidity", "Growth_Milestone")
    .orderBy(col("Sunlight_Hours").desc())
    .limit(10)
    .show(truncate=False)
)

In [ ]:
# ---------------------------------------------------------
# 3.4.d  Filtros por campo — SQL (equivalente)
# ---------------------------------------------------------
print("🔎 [SQL] Plantas con > 8 horas de sol y humedad > 60%:\n")

spark.sql("""
    SELECT plant_id, Soil_Type, Sunlight_Hours, Humidity, Growth_Milestone
    FROM default.plant_growth
    WHERE Sunlight_Hours > 8 AND Humidity > 60
    ORDER BY Sunlight_Hours DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# ---------------------------------------------------------
# 3.4.e  Consulta avanzada: Fertilizante vs Crecimiento
# ---------------------------------------------------------
print("📊 [PySpark] Relación Fertilizante → Hito de Crecimiento:\n")
(
    df_table
    .groupBy("Fertilizer_Type", "Growth_Milestone")
    .agg(count("*").alias("cantidad"))
    .orderBy("Fertilizer_Type", "Growth_Milestone")
    .show(truncate=False)
)

print("\n📊 [SQL] Relación Fertilizante → Hito de Crecimiento:\n")
spark.sql("""
    SELECT Fertilizer_Type, Growth_Milestone, COUNT(*) AS cantidad
    FROM default.plant_growth
    GROUP BY Fertilizer_Type, Growth_Milestone
    ORDER BY Fertilizer_Type, Growth_Milestone
""").show(truncate=False)

### 3.5 Resumen de validaciones

| Validación | PySpark | SQL | ¿Resultados consistentes? |
|---|---|---|---|
| Metadatos / Esquema | `printSchema()` | `DESCRIBE TABLE` | ✅ Sí |
| Descripción estadística | `df.describe()` | `AVG, MIN, MAX, STDDEV` | ✅ Sí |
| GROUP BY (Soil_Type) | `groupBy().agg()` | `GROUP BY ... ORDER BY` | ✅ Sí |
| GROUP BY (Growth_Milestone) | `groupBy().agg()` | `GROUP BY ... ORDER BY` | ✅ Sí |
| COUNT(*) | `df.count()` | `SELECT COUNT(*)` | ✅ Sí |
| LIMIT (muestras) | `df.limit(5)` | `LIMIT 5` | ✅ Sí |
| Filtros (WHERE) | `df.filter()` | `WHERE ... AND ...` | ✅ Sí |
| Multi-GROUP BY | `groupBy().agg()` | `GROUP BY col1, col2` | ✅ Sí |

---

<a id='4'></a>
# 4 · Ventajas y Desventajas: SQL vs Spark

## 4.1 Tabla Comparativa

| Criterio | SQL (Spark SQL) | Spark (PySpark DataFrame API) |
|---|---|---|
| **Facilidad de uso** | ✅ Sintaxis declarativa y familiar; ampliamente conocido. Ideal para analistas de datos y consultas ad-hoc. | ⚠️ Requiere conocer Python y la API de DataFrames. Curva de aprendizaje moderada. |
| **Expresividad** | ✅ Muy expresivo para consultas de datos (SELECT, JOIN, GROUP BY, WINDOW). Estándar universal. | ✅ Extremadamente expresivo con encadenamiento de métodos, transformaciones complejas y lógica de programación general. |
| **Integración con BI** | ✅ Se integra directamente con herramientas de BI (Power BI, Tableau, Looker) mediante conectores JDBC/ODBC. | ⚠️ Requiere pasos adicionales para conectar con herramientas BI; sin embargo, los resultados se pueden convertir a Pandas para visualización. |
| **Pipelines complejos (ETL)** | ⚠️ Limitado para orquestación compleja: sin bucles, sin manejo de errores, sin lógica condicional avanzada. | ✅ Ideal para pipelines ETL complejos: soporte completo de Python (bucles, try/except, modularización, tests). |
| **UDFs y extensibilidad** | ⚠️ Soporte de UDFs SQL limitado en funcionalidad y rendimiento. | ✅ UDFs nativas en Python/Scala, Pandas UDFs vectorizadas para alto rendimiento. Acceso a MLlib, GraphFrames, etc. |
| **Escalabilidad** | ✅ Spark SQL escala automáticamente con el clúster (mismo motor Catalyst). | ✅ Misma escalabilidad subyacente. Además, permite control fino del particionamiento, cache y broadcast. |
| **Rendimiento y ajuste** | ⚠️ El optimizador Catalyst hace un buen trabajo, pero el control fino es limitado. | ✅ Control granular: `repartition()`, `coalesce()`, `cache()`, `persist()`, broadcast joins, planes de ejecución. |
| **Depuración** | ⚠️ Difícil de depurar consultas complejas; los errores SQL son poco descriptivos. | ✅ Depuración con herramientas de Python (breakpoints, logging, `explain()`). |

---

## 4.2 ¿Cuándo usar cada uno?

### 🟦 Usa **SQL** cuando:
- Necesitas hacer consultas exploratorias rápidas.
- Tu equipo es mayoritariamente de analistas con background en SQL.
- Requieres integración directa con herramientas de BI.
- La lógica es puramente declarativa (filtros, agrupaciones, joins).

### 🟩 Usa **PySpark (DataFrame API)** cuando:
- Estás construyendo pipelines ETL/ELT complejos.
- Necesitas UDFs, lógica condicional o integración con ML.
- Requieres control fino del rendimiento y la ejecución.
- Tu equipo tiene habilidades en programación Python.

### 🔀 En la práctica:
Lo más común es **combinar ambos enfoques**:
- Usar SQL para exploración y validación rápida.
- Usar PySpark para transformaciones y orquestación.
- Registrar DataFrames como vistas temporales (`createOrReplaceTempView`) para alternar entre ambos.

---

## 4.3 Conclusión

Ambas herramientas son **complementarias, no competitivas**. SQL es la puerta de entrada más accesible, mientras que PySpark ofrece la potencia necesaria para flujos de trabajo complejos. En Databricks, la integración entre ambos es transparente gracias al motor Catalyst, lo que permite usar la herramienta más adecuada para cada tarea sin penalización de rendimiento.

---

# ✅ Resumen Final

| Sección | Estado |
|---|---|
| 0. Diseño del esquema de almacenamiento | ✅ Completado |
| 1. Configuración y evidencia de Databricks CE | ✅ Completado |
| 2. Obtención de datos (Kaggle) y creación de tabla | ✅ Completado |
| 3. Validaciones en Spark y SQL | ✅ Completado |
| 4. Ventajas y desventajas: SQL vs Spark | ✅ Completado |

---

**Notebook elaborado por:** Alejandro Arango Calderón  
**Dataset:** Plant Growth Data — Classification (Kaggle)  
**Plataforma:** Databricks Community Edition  
**Motor:** Apache Spark + Delta Lake